In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
  "answer": "The agentic loop keeps calling the model until it stops by wrapping the process of sending messages and running tools in a `while True` loop. The loop's core mechanism is as follows:\n\n1.  **Initialize**: The loop starts with an iteration counter and a flag, `has_function_calls`, set to `False`.\n2.  **Call the Model**: It sends the current message history (which includes instructions and the user's question, and previous model outputs/tool results) to the LLM.\n3.  **Process Response**: The model's response is received and appended to the message history. The loop then iterates through the items in the response:\n    *   **Function Call**: If an item is a `function_call`, the agent executes the corresponding tool (e.g., `search`) using the arguments provided by the model. The output of the tool call is then appended back to the message history, and the `has_function_calls` flag is set to `True`.\n    *   **Message**: If an item is a `message`, it means the model is pro

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xfd45f12f23e500b4c237b906470981da",
        "span_id": "0x0277a0ef559452b6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T14:07:39.640289Z",
    "end_time": "2026-07-20T14:07:39.640335Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "5f0af571-1f9a-40d0-aa4b-533a1e7fae24",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [7]:
from rag_helper import RAGBase
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

class RAGTraced(RAGBase):

    provider = TracerProvider()
    provider.add_span_processor(
        SimpleSpanProcessor(ConsoleSpanExporter())
    )
    trace.set_tracer_provider(provider)

    tracer = trace.get_tracer("llm-zoomcamp")

    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm"):
            return super().llm(prompt)

    def rag(self, query):
        with self.tracer.start_as_current_span("rag"):
            return super().rag(query)

In [8]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

In [9]:
from google import genai
client = genai.Client()
rag_traced = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x77af2749ab9ba6e2c31549e25ea110f2",
        "span_id": "0x4812d0f12b7389b4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xba2432d0d756accb",
    "start_time": "2026-07-20T14:36:43.565938Z",
    "end_time": "2026-07-20T14:36:43.568722Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "4efce7d5-fa0d-44a8-afa1-b01f622cc7c7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x77af2749ab9ba6e2c31549e25ea110f2",
        "span_id": "0xdf3cf16687f09836",
        "trace_state": "[]"
    },
    "kind": "SpanKind